# Deepfake Detection — Random Forest + Optuna Hyperparameter Optimization
ASVspoof 2019 LA dataset

Uses `rf_features` directly from the feature extraction pipeline: MFCC mean + std → (80,) vector.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.4 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve
)

import optuna
from optuna.samplers import TPESampler

from google.colab import drive

In [ ]:
# ── Cell 3: Mount Drive & load data ───────────────────────────────────────────
drive.mount('/content/drive')

# Using v2 pkl files — rf_features is now (80,): MFCC mean + std
TRAIN_PATH = ('/content/drive/MyDrive/ASV_Final_Features_train_v2.pkl', 'train')
DEV_PATH   = ('/content/drive/MyDrive/ASV_Final_Features_dev_v2.pkl',   'dev')
EVAL_PATH  = ('/content/drive/MyDrive/ASV_Final_Features_eval_v2.pkl',  'eval')

def load_dataset(path):
    df = pd.read_pickle(path[0])
    print(f'DataFrame loaded for {path[1]} set. '
          f'Total samples: {len(df)}, shape: {df.shape}')
    return df

train_df = load_dataset(TRAIN_PATH)
dev_df   = load_dataset(DEV_PATH)
eval_df  = load_dataset(EVAL_PATH)

print('\nLabel distribution:')
print('Train:', train_df['label'].value_counts().to_dict())
print('Dev:  ', dev_df['label'].value_counts().to_dict())
print('Eval: ', eval_df['label'].value_counts().to_dict())

Mounted at /content/drive
DataFrame loaded for train set. Total samples: 25380, shape: (25380, 5)
DataFrame loaded for dev set. Total samples: 24844, shape: (24844, 5)
DataFrame loaded for eval set. Total samples: 71237, shape: (71237, 5)

Label distribution:
Train: {1: 22800, 0: 2580}
Dev:   {1: 22296, 0: 2548}
Eval:  {1: 63882, 0: 7355}


In [ ]:
# ── Cell 4: Prepare features ───────────────────────────────────────────────────
# rf_features is already (80,): MFCC mean + std from feature extraction
X_train = np.stack(train_df['rf_features'].values)
X_dev   = np.stack(dev_df['rf_features'].values)
X_eval  = np.stack(eval_df['rf_features'].values)

y_train = train_df['label'].values
y_dev   = dev_df['label'].values
y_eval  = eval_df['label'].values

print(f'X_train shape : {X_train.shape}')  # expect (N, 80)
print(f'X_dev shape   : {X_dev.shape}')
print(f'X_eval shape  : {X_eval.shape}')

X_train shape : (25380, 80)
X_dev shape   : (24844, 80)
X_eval shape  : (71237, 80)


In [ ]:
# ── Cell 5: Scale features ─────────────────────────────────────────────────────
# Fit scaler on train only, transform all splits
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_dev_scaled   = scaler.transform(X_dev)
X_eval_scaled  = scaler.transform(X_eval)

print('Features scaled.')

Features scaled.


In [ ]:
# ── Cell 6: Evaluation function ────────────────────────────────────────────────
def evaluate_rf_model(model, X, y, dataset_name='Dataset', show_plots=True):
    """
    Returns (eer, fig).
    Pass show_plots=False during Optuna trials to suppress output.
    """
    y_pred   = model.predict(X)
    y_scores = model.predict_proba(X)[:, 0]

    acc = accuracy_score(y, y_pred)
    p   = precision_score(y, y_pred, zero_division=0)
    r   = recall_score(y, y_pred, zero_division=0)
    f1  = f1_score(y, y_pred, zero_division=0)

    fpr, tpr, _ = roc_curve(y, y_scores, pos_label=0)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute(fnr - fpr))] * 100

    if show_plots:
        print(f"\n{'='*10} {dataset_name} Results {'='*10}")
        print(f"EER:       {eer:.4f}%")
        print(f"Accuracy:  {acc:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"Precision: {p:.4f}")
        print(f"Recall:    {r:.4f}")
        print('\nClassification Report:')
        print(classification_report(y, y_pred, target_names=['Bonafide', 'Spoof']))

        cm = confusion_matrix(y, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Bonafide', 'Spoof'],
                    yticklabels=['Bonafide', 'Spoof'])
        ax.set_title(f'Confusion Matrix: {dataset_name}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        plt.tight_layout()
        plt.show()
        return eer, fig

    return eer, None

In [ ]:
# ── Cell 7: Optuna objective ───────────────────────────────────────────────────
def objective(trial):
    # ── Search space ───────────────────────────────────────────────────────────
    n_estimators     = trial.suggest_categorical('n_estimators',     [300, 500, 1000])
    max_depth        = trial.suggest_categorical('max_depth',        [10, 20, 30, None])
    min_samples_split= trial.suggest_int('min_samples_split',        2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf',         1, 4)
    max_features     = trial.suggest_categorical('max_features',     ['sqrt', 'log2', 0.5])
    bonafide_weight  = trial.suggest_categorical('bonafide_weight',  [3, 5, 7, 9])

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight={0: bonafide_weight, 1: 1},
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_scaled, y_train)

    eer, _ = evaluate_rf_model(rf, X_dev_scaled, y_dev, show_plots=False)
    return eer

In [ ]:
# ── Cell 8: Run Optuna study ───────────────────────────────────────────────────
STUDY_DB   = 'sqlite:////content/drive/MyDrive/optuna_rf.db'
STUDY_NAME = 'rf_asvspoof'

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STUDY_DB,
    load_if_exists=True,
    direction='minimize',
    sampler=TPESampler(seed=42, n_startup_trials=8),
)


study.optimize(
    objective,
    n_trials=50,
    timeout=2 * 3600,
    gc_after_trial=True,
)

print('\n── Optuna complete ──────────────────────────')
print(f'Best EER   : {study.best_value:.4f}%')
print(f'Best params: {study.best_params}')

[I 2026-04-07 02:38:28,877] A new study created in RDB with name: rf_asvspoof
[I 2026-04-07 02:40:38,114] Trial 0 finished with value: 15.966989594546108 and parameters: {'n_estimators': 500, 'max_depth': 10, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 0.5, 'bonafide_weight': 3}. Best is trial 0 with value: 15.966989594546108.
[I 2026-04-07 02:41:04,146] Trial 1 finished with value: 16.72048797990671 and parameters: {'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'bonafide_weight': 7}. Best is trial 0 with value: 15.966989594546108.
[I 2026-04-07 02:44:43,154] Trial 2 finished with value: 16.478292070326518 and parameters: {'n_estimators': 1000, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'bonafide_weight': 5}. Best is trial 0 with value: 15.966989594546108.
[I 2026-04-07 02:49:05,657] Trial 3 finished with value: 16.859526372443487 and parameters: {'n_estimators': 


── Optuna complete ──────────────────────────
Best EER   : 15.2225%
Best params: {'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bonafide_weight': 3}


In [ ]:
# ── Cell 9: Final model with best hyperparameters ─────────────────────────────
best = study.best_params
print('Training final model with:', best)

best_rf = RandomForestClassifier(
    n_estimators=best['n_estimators'],
    max_depth=best['max_depth'],
    min_samples_split=best['min_samples_split'],
    min_samples_leaf=best['min_samples_leaf'],
    max_features=best['max_features'],
    class_weight={0: best['bonafide_weight'], 1: 1},
    random_state=42,
    n_jobs=-1
)
best_rf.fit(X_train_scaled, y_train)
print('Final model trained.')

Training final model with: {'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bonafide_weight': 3}
Final model trained.


In [ ]:
# ── Cell 10: Evaluate on Dev set (with confusion matrix) ──────────────────────
dev_eer, dev_fig = evaluate_rf_model(
    best_rf, X_dev_scaled, y_dev, 'Development Set', show_plots=True
)


========== Development Set Results ==========
EER:       15.2225%
Accuracy:  0.9168
F1 Score:  0.9556
Precision: 0.9164
Recall:    0.9984

Classification Report:
              precision    recall  f1-score   support

    Bonafide       0.93      0.20      0.33      2548
       Spoof       0.92      1.00      0.96     22296

    accuracy                           0.92     24844
   macro avg       0.93      0.60      0.64     24844
weighted avg       0.92      0.92      0.89     24844



In [ ]:
# ── Cell 11: Evaluate on Eval set (with confusion matrix) ─────────────────────
eval_eer, eval_fig = evaluate_rf_model(
    best_rf, X_eval_scaled, y_eval, 'Evaluation Set', show_plots=True
)

print(f'\nGeneralization gap: {eval_eer - dev_eer:.2f}%')


========== Evaluation Set Results ==========
EER:       14.8242%
Accuracy:  0.9066
F1 Score:  0.9494
Precision: 0.9237
Recall:    0.9766

Classification Report:
              precision    recall  f1-score   support

    Bonafide       0.60      0.30      0.40      7355
       Spoof       0.92      0.98      0.95     63882

    accuracy                           0.91     71237
   macro avg       0.76      0.64      0.67     71237
weighted avg       0.89      0.91      0.89     71237


Generalization gap: -0.40%


In [ ]:
# ── Cell 12: Feature importances ──────────────────────────────────────────────
importances = best_rf.feature_importances_
feature_names = (
    [f'MFCC_{i+1}_mean' for i in range(40)] +
    [f'MFCC_{i+1}_std'  for i in range(40)]
)

indices = np.argsort(importances)[::-1][:20]  # top 20

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(20), importances[indices])
ax.set_xticks(range(20))
ax.set_xticklabels([feature_names[i] for i in indices], rotation=45, ha='right')
ax.set_title('Top 20 Feature Importances')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 13: Save model ────────────────────────────────────────────────────────
import pickle

with open('/content/drive/MyDrive/rf_best.pkl', 'wb') as f:
    pickle.dump(best_rf, f)

with open('/content/drive/MyDrive/rf_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model and scaler saved to Drive.')

Model and scaler saved to Drive.


In [ ]:
# ── Cell 14: Reload model ──────────────────────────────────────────────────────
import pickle

with open('/content/drive/MyDrive/rf_best.pkl', 'rb') as f:
    best_rf = pickle.load(f)

with open('/content/drive/MyDrive/rf_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print('Model and scaler reloaded.')

Model and scaler reloaded.
